In [ ]:
import pandas as pd, ast, numpy as np, glob, os

BASE = "/home/lillianchang/Documents/MOUS_hierarchical-representations"
SRC  = "/mnt/MOUSnew/SynologyDrive/source/SynologyDrive"

# ===== 1. MASTER TABLE: all MOUS words -> first & min bigram frequency =====
# Source: bigram_guslatho_allcounts.csv -- the canonical guslatho per-word table, which
# already carries each word's ordered 'Bigrams' and the matching 'Bigram Occurrence Counts'.
# (Words with no bigram, e.g. the single-letter 'u', are absent here and are simply not modelled.)
g = pd.read_csv(f"{BASE}/figures/bigram_guslatho_allcounts.csv")

def parse(x):
    try:
        return ast.literal_eval(x)
    except Exception:
        return []

g["_cnts"] = g["Bigram Occurrence Counts"].apply(parse)
g["First_Bigram_Count"] = g["_cnts"].apply(lambda c: c[0]   if len(c) > 0 else np.nan)
g["Min_Bigram_Count"]   = g["_cnts"].apply(lambda c: min(c) if len(c) > 0 else np.nan)
# log10(1 + count) convention, matching the existing first-bigram regressor pipeline
g["First_bigram_Zipf"]  = np.log10(1 + g["First_Bigram_Count"])
g["Min_bigram_Zipf"]    = np.log10(1 + g["Min_Bigram_Count"])

master = (g[["Word", "Bigrams", "First_Bigram_Count", "Min_Bigram_Count",
             "First_bigram_Zipf", "Min_bigram_Zipf"]]
          .dropna(subset=["First_bigram_Zipf", "Min_bigram_Zipf"])
          .drop_duplicates("Word").reset_index(drop=True))
master.to_csv(f"{BASE}/tables/mous_first_and_min_bigram_frequency.csv", index=False)
print("MASTER TABLE rows:", len(master))
master.head()

In [ ]:
# ===== 2. PER-SUBJECT regressor files (visual): full + restricted =====
# Merge the master table onto each visual subject's presented-word onset list.
master["_key"] = master["Word"].astype(str).str.strip().str.lower()
mlook = master.drop_duplicates("_key").set_index("_key")[["First_bigram_Zipf", "Min_bigram_Zipf"]]

files = sorted(glob.glob(f"{SRC}/sub-V*/func/*_bigrams_processed.csv"))
n_sub = 0
for fp in files:
    sub = os.path.basename(fp).split("_")[0]
    bp = pd.read_csv(fp)
    df = bp[["Onset", "Word"]].copy()
    df["_key"] = df["Word"].astype(str).str.strip().str.lower()
    df = df.merge(mlook, left_on="_key", right_index=True, how="left")
    # words absent from the master list get 0 (same handling as the original notebook)
    df["First_bigram_Zipf"] = df["First_bigram_Zipf"].fillna(0)
    df["Min_bigram_Zipf"]   = df["Min_bigram_Zipf"].fillna(0)
    out = df[["Onset", "Word", "First_bigram_Zipf", "Min_bigram_Zipf"]]
    out.to_csv(f"{SRC}/{sub}/func/{sub}_first_and_min_bigram.csv")
    # restricted set: only onsets whose first bigram is NOT the minimum-frequency bigram
    restricted = out[out["First_bigram_Zipf"] != out["Min_bigram_Zipf"]]
    restricted.to_csv(f"{SRC}/{sub}/func/{sub}_first_and_min_bigram_restricted.csv")
    n_sub += 1
print("subjects processed:", n_sub)

In [ ]:
# ===== 3. QC: collinearity + SPM-safety checks =====
corrs = []; bad = []
for fp in sorted(glob.glob(f"{SRC}/sub-V*/func/*_first_and_min_bigram.csv")):
    sub = os.path.basename(fp).split("_")[0]
    out = pd.read_csv(fp)
    sd_f, sd_m = out["First_bigram_Zipf"].std(), out["Min_bigram_Zipf"].std()
    if sd_f == 0 or sd_m == 0 or np.isnan(sd_f) or np.isnan(sd_m):
        bad.append(sub)            # a constant regressor would crash the SPM contrast
        continue
    corrs.append(out["First_bigram_Zipf"].corr(out["Min_bigram_Zipf"]))
print("mean within-subject corr(First, Min) bigram Zipf:", round(float(np.mean(corrs)), 3))
print("subjects with a constant/NaN regressor (exclude):", bad if bad else "none")